# Module 15 Lab: Tool Use Mastery & Enterprise Integration Patterns

**Mục tiêu lab:** Xây dựng LoanBot Tool Registry hoàn chỉnh với:
- Tool schema định nghĩa và validation (Pydantic)
- Agentic loop với tool calling simulation
- Parallel tool execution (asyncio.gather)
- Token Bucket rate limiting
- Circuit Breaker pattern
- Exponential backoff retry

**Nhân vật:** FinTech Corp / LoanBot v3.2 với 5 enterprise tools

**Hướng dẫn:** Chạy từng cell theo thứ tự. Section 7 là bài tập tự thực hành.

## Section 1: Setup & Mock Data

In [ ]:
import asyncio
import hashlib
import time
import random
import json
from dataclasses import dataclass, field
from typing import Dict, List, Optional, Any, Callable, Literal
from enum import Enum

# 4 mock LoanBot customers
CUSTOMERS = {
    'FC-2024-001': {
        'name': 'Nguyễn Văn An',
        'id_number': '001234567890',
        'amount_vnd': 500_000_000,
        'purpose': 'home_purchase',
        'property_address': 'Quận 1, TP.HCM',
        'monthly_income': 45_000_000,
    },
    'FC-2024-002': {
        'name': 'Trần Thị Bình',
        'id_number': '002345678901',
        'amount_vnd': 200_000_000,
        'purpose': 'business',
        'property_address': None,
        'monthly_income': 18_000_000,
    },
    'FC-2024-003': {
        'name': 'Lê Văn Cường',
        'id_number': '003456789012',
        'amount_vnd': 100_000_000,
        'purpose': 'personal',
        'property_address': None,
        'monthly_income': 8_000_000,
    },
    'FC-2024-004': {
        'name': 'Phạm Thị Dung',
        'id_number': '004567890123',
        'amount_vnd': 350_000_000,
        'purpose': 'home_purchase',
        'property_address': 'Quận 7, TP.HCM',
        'monthly_income': 32_000_000,
    },
}

# Mock external tool responses
MOCK_CREDIT_SCORES = {
    'FC-2024-001': {'credit_score': 720, 'credit_grade': 'A', 'payment_history': 'PPPPPPPPPPPPPPPPPPPPPPPP', 'derogatory_marks': 0, 'active_loans': 1},
    'FC-2024-002': {'credit_score': 580, 'credit_grade': 'C', 'payment_history': 'PPLPPPPPLPPPPPPPPPLPPPPP', 'derogatory_marks': 1, 'active_loans': 2},
    'FC-2024-003': {'credit_score': 400, 'credit_grade': 'D', 'payment_history': 'PPLLPPLLPPLLPPLLPPLLPPLL', 'derogatory_marks': 3, 'active_loans': 4},
    'FC-2024-004': {'credit_score': 645, 'credit_grade': 'B', 'payment_history': 'PPPPPLPPPPPPPPPPLPPPPPPP', 'derogatory_marks': 1, 'active_loans': 1},
}

MOCK_COMPLIANCE = {
    'FC-2024-001': {'aml_clear': True,  'pep_match': False, 'blacklisted': False, 'risk_level': 'LOW'},
    'FC-2024-002': {'aml_clear': True,  'pep_match': False, 'blacklisted': False, 'risk_level': 'MEDIUM'},
    'FC-2024-003': {'aml_clear': False, 'pep_match': False, 'blacklisted': True,  'risk_level': 'HIGH'},
    'FC-2024-004': {'aml_clear': True,  'pep_match': False, 'blacklisted': False, 'risk_level': 'LOW'},
}

MOCK_INCOME = {
    'FC-2024-001': {'verified_income': 45_000_000, 'income_source': 'employment', 'employer': 'Vingroup', 'verified': True},
    'FC-2024-002': {'verified_income': 16_500_000, 'income_source': 'business',    'employer': 'Self',     'verified': True},
    'FC-2024-003': {'verified_income': 7_200_000,  'income_source': 'employment', 'employer': 'SME',      'verified': True},
    'FC-2024-004': {'verified_income': 31_000_000, 'income_source': 'employment', 'employer': 'FPT',      'verified': True},
}

MOCK_PROPERTY = {
    'Quận 1, TP.HCM':  {'estimated_value_vnd': 8_500_000_000, 'ltv_ratio': 0.059, 'market_trend': 'UP'},
    'Quận 7, TP.HCM':  {'estimated_value_vnd': 4_200_000_000, 'ltv_ratio': 0.083, 'market_trend': 'STABLE'},
}

print('✅ Mock data loaded: 4 customers, 4 mock external services')

## Section 2: Tool Schema & Pydantic Validation

In [ ]:
# ─── Tool JSON Schemas (sent to Claude API) ───────────────────────────────────

TOOL_SCHEMAS = [
    {
        'name': 'get_credit_score',
        'description': 'Lấy điểm tín dụng và lịch sử 24 tháng từ CIC NHNN.',
        'input_schema': {
            'type': 'object',
            'properties': {
                'customer_id':     {'type': 'string', 'pattern': r'^FC-\d{4}-\d{3,}$',
                                    'description': 'Mã khách hàng format FC-YYYY-NNN'},
                'id_number':       {'type': 'string', 'pattern': r'^\d{12}$',
                                    'description': 'CCCD 12 chữ số'},
                'include_history': {'type': 'boolean', 'default': True,
                                    'description': 'Lấy 24 tháng payment history'},
            },
            'required': ['customer_id', 'id_number']
        }
    },
    {
        'name': 'check_compliance',
        'description': 'AML/PEP/blacklist check. CRITICAL — phải pass trước khi approve.',
        'input_schema': {
            'type': 'object',
            'properties': {
                'customer_id': {'type': 'string', 'pattern': r'^FC-\d{4}-\d{3,}$'},
                'id_number':   {'type': 'string', 'pattern': r'^\d{12}$'},
                'loan_amount': {'type': 'number', 'minimum': 0, 'description': 'VNĐ'},
            },
            'required': ['customer_id', 'id_number']
        }
    },
    {
        'name': 'verify_income',
        'description': 'Xác minh thu nhập từ thuế và employer database.',
        'input_schema': {
            'type': 'object',
            'properties': {
                'customer_id': {'type': 'string', 'pattern': r'^FC-\d{4}-\d{3,}$'},
                'declared_income': {'type': 'number', 'minimum': 0, 'description': 'Thu nhập khai báo (VNĐ/tháng)'},
            },
            'required': ['customer_id', 'declared_income']
        }
    },
    {
        'name': 'get_property_valuation',
        'description': 'Định giá tài sản thế chấp từ địa chỉ. Optional — chỉ dùng khi có collateral.',
        'input_schema': {
            'type': 'object',
            'properties': {
                'address':       {'type': 'string', 'description': 'Địa chỉ tài sản'},
                'property_type': {'type': 'string', 'enum': ['apartment', 'house', 'land'], 'default': 'apartment'},
            },
            'required': ['address']
        }
    },
]

print('Tool schemas defined:')
for t in TOOL_SCHEMAS:
    required = t['input_schema'].get('required', [])
    props    = list(t['input_schema']['properties'].keys())
    print(f"  🔧 {t['name']}: required={required}, optional={[p for p in props if p not in required]}")

In [ ]:
# ─── Pydantic-style validation (pure Python dataclass simulation) ─────────────
# (Using manual validation to avoid requiring pydantic install)

import re

class ValidationError(Exception):
    pass

def validate_customer_id(cid: str) -> str:
    if not re.match(r'^FC-\d{4}-\d{3,}$', cid):
        raise ValidationError(f'Invalid customer_id: {cid!r}. Expected format: FC-YYYY-NNN')
    return cid

def validate_id_number(idn: str) -> str:
    if not re.match(r'^\d{12}$', str(idn)):
        raise ValidationError(f'Invalid id_number: {idn!r}. Expected 12 digits.')
    return str(idn)

@dataclass
class CreditBureauInput:
    customer_id:     str
    id_number:       str
    include_history: bool = True

    def __post_init__(self):
        self.customer_id = validate_customer_id(self.customer_id)
        self.id_number   = validate_id_number(self.id_number)

@dataclass
class LoanDecisionOutput:
    loan_id:         str
    decision:        str  # APPROVE | REJECT | REVIEW | HITL
    risk_score:      float  # 0.0–1.0
    confidence:      float
    key_factors:     List[str]
    regulation_refs: List[str]
    hitl_required:   bool
    hitl_level:      Optional[int] = None  # 1-3

    def __post_init__(self):
        if self.decision not in ('APPROVE', 'REJECT', 'REVIEW', 'HITL'):
            raise ValidationError(f'Invalid decision: {self.decision}')
        if not 0.0 <= self.risk_score <= 1.0:
            raise ValidationError(f'risk_score out of range: {self.risk_score}')
        if self.hitl_level is not None and self.hitl_level not in (1, 2, 3):
            raise ValidationError(f'hitl_level must be 1-3: {self.hitl_level}')

# Test validation
print('=== Testing Input Validation ===')
try:
    inp = CreditBureauInput(customer_id='FC-2024-004', id_number='004567890123')
    print(f'✅ Valid: {inp}')
except ValidationError as e:
    print(f'❌ Error: {e}')

# Test invalid inputs
test_cases = [
    ('FC-2024-004', '12345'),       # id_number too short
    ('FC2024004',   '004567890123'), # customer_id wrong format
    ('FC-2024-004', 'abcdefghijkl'), # id_number not digits
]
print('\n=== Testing Invalid Inputs ===')
for cid, idn in test_cases:
    try:
        CreditBureauInput(customer_id=cid, id_number=idn)
        print(f'  ⚠️  Should have failed: ({cid}, {idn})')
    except ValidationError as e:
        print(f'  ✅ Caught: {e}')

# Test output validation
print('\n=== Testing Output Validation ===')
try:
    out = LoanDecisionOutput(
        loan_id='FC-2024-004', decision='APPROVE', risk_score=0.28,
        confidence=0.87, key_factors=['credit_score_645', 'income_verified'],
        regulation_refs=['TT39 Điều 15', 'QĐ493 Khoản 2'],
        hitl_required=False
    )
    print(f'✅ Valid output: {out.decision}, risk={out.risk_score}')
except ValidationError as e:
    print(f'❌ Error: {e}')

## Section 3: Tool Implementations with Caching & Rate Limiting

In [ ]:
# ─── Token Bucket Rate Limiter ────────────────────────────────────────────────

class TokenBucketRateLimiter:
    """Token bucket: allows burst up to capacity, refills at rate_rps/s."""

    def __init__(self, rate_rps: int, capacity: int = 10):
        self.rate      = rate_rps
        self.capacity  = capacity
        self.tokens    = float(capacity)
        self.last_tick = time.monotonic()
        self._calls    = 0
        self._rejected = 0

    def acquire(self) -> bool:
        now   = time.monotonic()
        delta = now - self.last_tick
        self.tokens    = min(self.capacity, self.tokens + delta * self.rate)
        self.last_tick = now
        if self.tokens >= 1:
            self.tokens -= 1
            self._calls += 1
            return True
        self._rejected += 1
        return False

    def stats(self) -> dict:
        return {'allowed': self._calls, 'rejected': self._rejected,
                'tokens_remaining': round(self.tokens, 2)}


# ─── Tool Config ─────────────────────────────────────────────────────────────

@dataclass
class ToolConfig:
    name:           str
    ttl_seconds:    int   = 0        # 0 = no cache
    rate_limit_rps: int   = 10
    timeout_s:      float = 5.0
    is_write:       bool  = False
    is_critical:    bool  = False    # CRITICAL = pipeline blocks if fail
    simulated_delay:float = 0.0      # Mock latency

TOOL_CONFIGS = {
    'get_credit_score':      ToolConfig('get_credit_score',      ttl_seconds=86_400,    rate_limit_rps=5,  simulated_delay=1.2, is_critical=True),
    'check_compliance':      ToolConfig('check_compliance',      ttl_seconds=0,         rate_limit_rps=20, simulated_delay=0.6, is_critical=True),
    'verify_income':         ToolConfig('verify_income',         ttl_seconds=2_592_000, rate_limit_rps=3,  simulated_delay=1.5),
    'get_property_valuation':ToolConfig('get_property_valuation',ttl_seconds=604_800,   rate_limit_rps=2,  simulated_delay=2.1),
    'send_notification':     ToolConfig('send_notification',     ttl_seconds=0,         rate_limit_rps=50, simulated_delay=0.2, is_write=True),
}


# ─── Tool Implementations (mock) ─────────────────────────────────────────────

def impl_credit_score(customer_id: str, id_number: str, include_history: bool = True) -> dict:
    validate_customer_id(customer_id)
    if customer_id not in MOCK_CREDIT_SCORES:
        raise KeyError(f'Customer {customer_id} not found in CIC')
    result = dict(MOCK_CREDIT_SCORES[customer_id])
    result['retrieved_at'] = '2026-06-26T10:00:00Z'
    return result

def impl_compliance(customer_id: str, id_number: str, loan_amount: float = 0) -> dict:
    validate_customer_id(customer_id)
    return dict(MOCK_COMPLIANCE[customer_id])

def impl_income(customer_id: str, declared_income: float) -> dict:
    data = dict(MOCK_INCOME[customer_id])
    data['discrepancy_pct'] = round(abs(data['verified_income'] - declared_income) / declared_income * 100, 1)
    return data

def impl_property(address: str, property_type: str = 'apartment') -> dict:
    if address not in MOCK_PROPERTY:
        return {'estimated_value_vnd': None, 'error': 'Address not in database'}
    return dict(MOCK_PROPERTY[address])

def impl_notification(customer_id: str, channel: str, message: str) -> dict:
    return {'status': 'sent', 'channel': channel, 'customer_id': customer_id}

TOOL_IMPLS = {
    'get_credit_score':       impl_credit_score,
    'check_compliance':       impl_compliance,
    'verify_income':          impl_income,
    'get_property_valuation': impl_property,
    'send_notification':      impl_notification,
}

print('✅ Tool implementations registered')
print(f'   Critical tools: {[name for name, cfg in TOOL_CONFIGS.items() if cfg.is_critical]}')
print(f'   Write tools:    {[name for name, cfg in TOOL_CONFIGS.items() if cfg.is_write]}')

In [ ]:
# ─── Tool Registry ─────────────────────────────────────────────────────────────

class ToolRegistry:
    """Central registry với caching, rate limiting, idempotency cho tất cả tools."""

    def __init__(self):
        self._cache:   Dict[str, tuple] = {}     # key → (value, expire_time)
        self._limiters = {
            name: TokenBucketRateLimiter(cfg.rate_limit_rps)
            for name, cfg in TOOL_CONFIGS.items()
        }
        self._idempotency_set: set = set()
        self._call_log: List[dict] = []

    def call(self, tool_name: str, params: dict) -> dict:
        cfg  = TOOL_CONFIGS[tool_name]
        impl = TOOL_IMPLS[tool_name]

        # Idempotency for write tools
        if cfg.is_write:
            idem_key = hashlib.md5(f'{tool_name}:{sorted(params.items())}'.encode()).hexdigest()
            if idem_key in self._idempotency_set:
                self._log(tool_name, params, 'IDEMPOTENT_SKIP')
                return {'status': 'skipped', 'reason': 'already_sent'}
            self._idempotency_set.add(idem_key)

        # Cache check
        cache_key = f'{tool_name}:{tuple(sorted(params.items()))}'
        if cfg.ttl_seconds > 0:
            cached = self._get_cache(cache_key)
            if cached is not None:
                self._log(tool_name, params, 'CACHE_HIT')
                return cached

        # Rate limit
        if not self._limiters[tool_name].acquire():
            raise RuntimeError(f'{tool_name}: rate limit exceeded (>{cfg.rate_limit_rps} rps)')

        # Execute
        start = time.time()
        result = impl(**params)
        elapsed = time.time() - start
        self._log(tool_name, params, 'EXECUTED', elapsed)

        # Cache store
        if cfg.ttl_seconds > 0:
            self._set_cache(cache_key, result, cfg.ttl_seconds)

        return result

    def _get_cache(self, key: str) -> Optional[Any]:
        if key in self._cache:
            val, exp = self._cache[key]
            if time.time() < exp:
                return val
            del self._cache[key]
        return None

    def _set_cache(self, key: str, val: Any, ttl: int):
        self._cache[key] = (val, time.time() + ttl)

    def _log(self, tool: str, params: dict, status: str, elapsed: float = 0):
        self._call_log.append({'tool': tool, 'status': status, 'elapsed_ms': round(elapsed*1000)})

    def print_stats(self):
        print('\n=== Tool Registry Stats ===')
        for tool_name, limiter in self._limiters.items():
            stats = limiter.stats()
            print(f'  {tool_name}: allowed={stats["allowed"]}, rejected={stats["rejected"]}')
        print(f'  Cache entries: {len(self._cache)}')
        print(f'  Idempotency keys: {len(self._idempotency_set)}')


# Demo: Basic tool calls
registry = ToolRegistry()
print('=== Basic Tool Calls ===')

credit = registry.call('get_credit_score', {'customer_id': 'FC-2024-004', 'id_number': '004567890123'})
print(f'Credit FC-2024-004: score={credit["credit_score"]}, grade={credit["credit_grade"]}')

compliance = registry.call('check_compliance', {'customer_id': 'FC-2024-004', 'id_number': '004567890123'})
print(f'Compliance FC-2024-004: aml_clear={compliance["aml_clear"]}, risk={compliance["risk_level"]}')

# Second call — should hit cache for credit score
credit2 = registry.call('get_credit_score', {'customer_id': 'FC-2024-004', 'id_number': '004567890123'})
print(f'Credit (2nd call): {credit2["credit_score"]} [should be CACHE_HIT]')

# Idempotency test for notification
n1 = registry.call('send_notification', {'customer_id': 'FC-2024-004', 'channel': 'SMS', 'message': 'Hồ sơ đang xử lý'})
n2 = registry.call('send_notification', {'customer_id': 'FC-2024-004', 'channel': 'SMS', 'message': 'Hồ sơ đang xử lý'})
print(f'\nNotification 1: {n1["status"]}')
print(f'Notification 2 (duplicate): {n2["status"]} [{n2.get("reason", "")}]')

registry.print_stats()

## Section 4: Parallel Tool Execution

In [ ]:
# ─── Simulate tool_use blocks (as would come from Claude API) ─────────────────

@dataclass
class ToolUseBlock:
    """Simulates Claude API tool_use content block."""
    id:    str
    name:  str
    input: dict

@dataclass
class ToolResultBlock:
    """Simulates Claude API tool_result content block for user message."""
    type:        str = 'tool_result'
    tool_use_id: str = ''
    content:     str = ''
    is_error:    bool = False


async def execute_tools_parallel(tool_uses: List[ToolUseBlock], reg: ToolRegistry) -> tuple:
    """Execute multiple tool_use blocks in parallel, collect results and timing."""

    async def _run_one(tu: ToolUseBlock) -> tuple:
        cfg = TOOL_CONFIGS.get(tu.name)
        delay = cfg.simulated_delay if cfg else 0.0
        await asyncio.sleep(delay)  # simulate network latency
        try:
            result = reg.call(tu.name, tu.input)
            return ToolResultBlock(tool_use_id=tu.id, content=json.dumps(result)), delay
        except Exception as e:
            return ToolResultBlock(tool_use_id=tu.id, content=f'ERROR: {e}', is_error=True), delay

    start = time.time()
    tasks = [_run_one(tu) for tu in tool_uses]
    raw   = await asyncio.gather(*tasks, return_exceptions=True)
    elapsed = time.time() - start

    results = []
    for item in raw:
        if isinstance(item, Exception):
            results.append(ToolResultBlock(content=f'ERROR: {item}', is_error=True))
        else:
            results.append(item[0])

    return results, elapsed


# ─── Demo: Sequential vs Parallel ────────────────────────────────────────────

# Simulate Claude's tool_use blocks for FC-2024-004 assessment
tool_uses_004 = [
    ToolUseBlock(id='tu_001', name='get_credit_score',
                 input={'customer_id': 'FC-2024-004', 'id_number': '004567890123'}),
    ToolUseBlock(id='tu_002', name='check_compliance',
                 input={'customer_id': 'FC-2024-004', 'id_number': '004567890123'}),
    ToolUseBlock(id='tu_003', name='verify_income',
                 input={'customer_id': 'FC-2024-004', 'declared_income': 32_000_000}),
    ToolUseBlock(id='tu_004', name='get_property_valuation',
                 input={'address': 'Quận 7, TP.HCM', 'property_type': 'apartment'}),
]

reg2 = ToolRegistry()

# Sequential (for comparison)
print('=== Sequential Execution ===')
seq_start = time.time()
seq_results = []
for tu in tool_uses_004:
    cfg = TOOL_CONFIGS.get(tu.name)
    time.sleep(cfg.simulated_delay if cfg else 0)
    result = reg2.call(tu.name, tu.input)
    seq_results.append(result)
seq_elapsed = time.time() - seq_start
print(f'Sequential time: {seq_elapsed:.2f}s')
print(f'Expected (sum): {sum(TOOL_CONFIGS[tu.name].simulated_delay for tu in tool_uses_004):.1f}s')

# Parallel
print('\n=== Parallel Execution ===')
reg3 = ToolRegistry()
par_results, par_elapsed = asyncio.run(execute_tools_parallel(tool_uses_004, reg3))
print(f'Parallel time: {par_elapsed:.2f}s')
expected_par = max(TOOL_CONFIGS[tu.name].simulated_delay for tu in tool_uses_004)
print(f'Expected (max): {expected_par:.1f}s')
print(f'Speedup: {seq_elapsed / par_elapsed:.1f}x')

print(f'\nResults ({len(par_results)} tool results):')
for i, (tu, r) in enumerate(zip(tool_uses_004, par_results)):
    status = '❌ ERROR' if r.is_error else '✅ OK'
    data = json.loads(r.content) if not r.is_error else {}
    preview = str(data)[:60] + '...' if len(str(data)) > 60 else str(data)
    print(f'  {status} {tu.name}: {preview}')

## Section 5: Circuit Breaker & Exponential Backoff

In [ ]:
# ─── Circuit Breaker ───────────────────────────────────────────────────────────

class CBState(Enum):
    CLOSED    = 'CLOSED'
    OPEN      = 'OPEN'
    HALF_OPEN = 'HALF_OPEN'

class CircuitBreaker:
    """Circuit breaker cho LoanBot tools."""

    def __init__(self, tool_name: str, failure_threshold: int = 5, cooldown_s: float = 2.0):
        self.tool_name         = tool_name
        self.failure_threshold = failure_threshold
        self.cooldown_s        = cooldown_s
        self.state             = CBState.CLOSED
        self.failure_count     = 0
        self.opened_at: Optional[float] = None
        self.history: List[str] = []

    def call(self, fn: Callable, *args, **kwargs):
        if self.state == CBState.OPEN:
            if time.time() - self.opened_at >= self.cooldown_s:
                self.state = CBState.HALF_OPEN
                self.history.append('TRANSITION → HALF_OPEN')
            else:
                remaining = self.cooldown_s - (time.time() - self.opened_at)
                raise RuntimeError(f'Circuit OPEN for {self.tool_name} ({remaining:.1f}s remaining)')
        try:
            result = fn(*args, **kwargs)
            if self.state == CBState.HALF_OPEN:
                self._reset()
                self.history.append('TRANSITION → CLOSED (trial success)')
            else:
                self.history.append('CALL_OK')
            return result
        except Exception as e:
            self._on_failure()
            raise

    def _on_failure(self):
        self.failure_count += 1
        self.history.append(f'FAILURE ({self.failure_count}/{self.failure_threshold})')
        if self.failure_count >= self.failure_threshold or self.state == CBState.HALF_OPEN:
            self.state     = CBState.OPEN
            self.opened_at = time.time()
            self.history.append('TRANSITION → OPEN')

    def _reset(self):
        self.state         = CBState.CLOSED
        self.failure_count = 0
        self.opened_at     = None

    def status(self) -> str:
        return f'{self.tool_name}: {self.state.value} (failures={self.failure_count})'


# Demo: Circuit Breaker behavior
print('=== Circuit Breaker Demo ===')

def flaky_credit_api(customer_id: str, id_number: str, include_history: bool = True):
    """Simulates a flaky CIC API that fails 60% of the time."""
    if random.random() < 0.6:
        raise ConnectionError('CIC API timeout')
    return MOCK_CREDIT_SCORES.get(customer_id, {})

cb = CircuitBreaker('get_credit_score', failure_threshold=3, cooldown_s=1.0)

random.seed(42)  # reproducible
for i in range(10):
    try:
        result = cb.call(flaky_credit_api, 'FC-2024-004', '004567890123')
        print(f'  Call {i+1}: ✅ OK — score={result.get("credit_score", "N/A")} | {cb.status()}')
    except RuntimeError as e:
        print(f'  Call {i+1}: 🚫 BLOCKED — {e}')
    except ConnectionError as e:
        print(f'  Call {i+1}: ❌ FAILED — {e} | {cb.status()}')
    time.sleep(0.1)

# Wait for cooldown, then try again (HALF-OPEN test)
print(f'\n  Waiting {cb.cooldown_s}s for cooldown...')
time.sleep(cb.cooldown_s)

# Use reliable function for trial
try:
    result = cb.call(impl_credit_score, 'FC-2024-004', '004567890123')
    print(f'  Trial call: ✅ OK — {cb.status()}')
except Exception as e:
    print(f'  Trial call: ❌ — {e} | {cb.status()}')

print(f'\nCircuit history: {cb.history}')

In [ ]:
# ─── Exponential Backoff Retry ─────────────────────────────────────────────────

def with_retry(fn: Callable, max_attempts: int = 4,
               base_delay: float = 0.1, max_delay: float = 2.0,
               retryable_exceptions=(ConnectionError, TimeoutError)) -> Callable:
    """Wraps fn with exponential backoff retry. Returns wrapped function."""

    def wrapper(*args, **kwargs):
        attempt  = 0
        delays   = []
        while attempt < max_attempts:
            try:
                return fn(*args, **kwargs)
            except retryable_exceptions as e:
                attempt += 1
                if attempt >= max_attempts:
                    print(f'    [retry] All {max_attempts} attempts exhausted → giving up')
                    raise
                delay = min(max_delay, base_delay * (2 ** attempt) + random.uniform(0, 0.05))
                delays.append(round(delay, 3))
                print(f'    [retry] Attempt {attempt}/{max_attempts} failed ({e}). Retry in {delay:.3f}s')
                time.sleep(delay)
        raise RuntimeError('Unreachable')

    return wrapper


# Demo
print('=== Exponential Backoff Demo ===')

call_count = [0]
def sometimes_fails(customer_id: str, id_number: str, include_history: bool = True):
    call_count[0] += 1
    # Fails first 2 attempts, succeeds on 3rd
    if call_count[0] <= 2:
        raise ConnectionError(f'Connection refused (attempt {call_count[0]})')
    return MOCK_CREDIT_SCORES[customer_id]

resilient_credit = with_retry(sometimes_fails, max_attempts=4, base_delay=0.05)

start = time.time()
try:
    result = resilient_credit('FC-2024-004', '004567890123')
    print(f'✅ Success on attempt {call_count[0]}: score={result["credit_score"]}')
    print(f'   Total time: {time.time()-start:.3f}s')
except Exception as e:
    print(f'❌ Failed: {e}')

# Demo: exhausting all retries
print('\n--- Always Failing API ---')
def always_fails(**kwargs):
    raise ConnectionError('Service Down')

resilient_bad = with_retry(always_fails, max_attempts=3, base_delay=0.02)
try:
    resilient_bad(customer_id='FC-2024-004', id_number='004567890123')
except ConnectionError as e:
    print(f'✅ Correctly raised after exhausting retries: {e}')

## Section 6: Full Agentic Loop Simulation

In [ ]:
# ─── Agentic Loop Simulation ──────────────────────────────────────────────────
# Simulates what Claude API would do for LoanBot assessment

@dataclass
class SimulatedMessage:
    role:    str  # 'user' | 'assistant'
    content: Any  # str or list of blocks

def mock_claude_response(messages: list, customer_data: dict) -> dict:
    """
    Simulates Claude API response.
    Turn 1: requests tools.
    Turn 2+: generates final assessment.
    """
    turn = len([m for m in messages if m['role'] == 'assistant']) + 1

    if turn == 1:
        # Claude decides what tools to call
        tool_uses = [
            ToolUseBlock('tu_a', 'get_credit_score',
                         {'customer_id': customer_data['customer_id'],
                          'id_number': customer_data['id_number']}),
            ToolUseBlock('tu_b', 'check_compliance',
                         {'customer_id': customer_data['customer_id'],
                          'id_number': customer_data['id_number']}),
            ToolUseBlock('tu_c', 'verify_income',
                         {'customer_id': customer_data['customer_id'],
                          'declared_income': customer_data['monthly_income']}),
        ]
        if customer_data.get('property_address'):
            tool_uses.append(ToolUseBlock('tu_d', 'get_property_valuation',
                                          {'address': customer_data['property_address']}))
        return {'stop_reason': 'tool_use', 'tool_uses': tool_uses}

    else:
        # Extract tool results from messages to simulate assessment
        tool_data = {}
        for msg in messages:
            if msg['role'] == 'user' and isinstance(msg.get('content'), list):
                for block in msg['content']:
                    if isinstance(block, ToolResultBlock) and not block.is_error:
                        data = json.loads(block.content)
                        tool_data.update(data)

        score     = tool_data.get('credit_score', 500)
        blacklist = tool_data.get('blacklisted', False)
        aml_clear = tool_data.get('aml_clear', True)
        income_ok = tool_data.get('verified', False)

        if blacklist or not aml_clear:
            decision = 'REJECT'
        elif score >= 700:
            decision = 'APPROVE'
        elif score >= 600:
            decision = 'REVIEW'
        else:
            decision = 'REJECT'

        assessment_text = (
            f'Kết quả thẩm định:\n'
            f'- Credit Score: {score} → {tool_data.get("credit_grade", "N/A")}\n'
            f'- Compliance: AML={aml_clear}, Blacklist={blacklist}\n'
            f'- Income verified: {income_ok}\n'
            f'- QUYẾT ĐỊNH: {decision}\n'
            f'- Cơ sở pháp lý: TT39/2016 Điều 15'
        )
        return {'stop_reason': 'end_turn', 'text': assessment_text, 'decision': decision}


async def run_loanbot_agentic_loop(customer_id: str, max_iterations: int = 5) -> dict:
    """Full agentic loop cho một customer với parallel tool execution."""

    cust = CUSTOMERS[customer_id]
    cust['customer_id'] = customer_id

    messages = [{'role': 'user', 'content': f'Thẩm định hồ sơ {customer_id}: {cust}'}]
    reg      = ToolRegistry()
    iteration = 0
    start_time = time.time()

    print(f'\n=== Agentic Loop: {customer_id} ({cust["name"]}) ===')

    while iteration < max_iterations:
        iteration += 1
        response = mock_claude_response(messages, cust)

        if response['stop_reason'] == 'end_turn':
            total = time.time() - start_time
            print(f'  Iterations: {iteration} | Total time: {total:.2f}s')
            print(f'  Decision: {response["decision"]}')
            return {'decision': response['decision'], 'text': response['text'],
                    'iterations': iteration, 'time_s': round(total, 2)}

        elif response['stop_reason'] == 'tool_use':
            tool_uses = response['tool_uses']
            print(f'  Iter {iteration}: Claude requests {len(tool_uses)} tools in parallel')
            for tu in tool_uses:
                delay = TOOL_CONFIGS.get(tu.name, ToolConfig('x')).simulated_delay
                print(f'    → {tu.name} ({delay}s simulated latency)')

            results, elapsed = await execute_tools_parallel(tool_uses, reg)
            errors = [r for r in results if r.is_error]

            print(f'  Parallel execution: {elapsed:.2f}s | Errors: {len(errors)}')

            # Check critical tool failures
            critical_errors = [
                (tu, r) for tu, r in zip(tool_uses, results)
                if r.is_error and TOOL_CONFIGS.get(tu.name, ToolConfig('x')).is_critical
            ]
            if critical_errors:
                print(f'  ⚠️  Critical tool failure → HITL escalation')
                return {'decision': 'HITL', 'reason': 'critical_tool_failure'}

            # Add assistant turn (tool_use blocks) + user turn (tool_results)
            messages.append({'role': 'assistant', 'content': tool_uses})
            messages.append({'role': 'user', 'content': results})

    return {'decision': 'HITL', 'reason': 'max_iterations_exceeded'}


# Run loop for all 4 customers
results = {}
for cid in ['FC-2024-001', 'FC-2024-002', 'FC-2024-003', 'FC-2024-004']:
    result = asyncio.run(run_loanbot_agentic_loop(cid))
    results[cid] = result

print('\n=== Summary ===')
for cid, r in results.items():
    print(f'{cid}: {r["decision"]} ({r.get("time_s", "N/A")}s)')

## Section 7: Bài tập tự thực hành — Tool Priority & Selective Calling

In [ ]:
# ─── BÀI TẬP: Implement smart_tool_selector ────────────────────────────────────
#
# LoanBot đang gọi tất cả 4 tools cho mọi application.
# Nhiệm vụ: Implement smart_tool_selector() chỉ gọi tools thực sự cần thiết:
#
# Rules:
#   1. LUÔN gọi: get_credit_score, check_compliance (critical tools)
#   2. Chỉ gọi verify_income nếu declared_income > 0
#   3. Chỉ gọi get_property_valuation nếu property_address != None
#   4. Nếu compliance blacklisted=True từ cache → KHÔNG cần gọi thêm tools nào
#
# Expected output:
#   FC-2024-001: 4 tools (has property, has income)
#   FC-2024-002: 3 tools (no property)
#   FC-2024-003: 2-3 tools (blacklisted — stop after compliance)
#   FC-2024-004: 4 tools (has property, has income)

def smart_tool_selector(customer_id: str, customer_data: dict, reg: ToolRegistry) -> List[ToolUseBlock]:
    """
    TODO: Implement smart tool selection.
    Return: list of ToolUseBlocks to execute.
    """
    tool_uses = []

    # --- TODO: Implement rules 1-4 ---
    # Gợi ý:
    #   - Rule 1: luôn thêm get_credit_score và check_compliance
    #   - Rule 4: kiểm tra cache cho compliance trước (gọi reg._get_cache)
    #   - Rule 2: check customer_data['monthly_income'] > 0
    #   - Rule 3: check customer_data.get('property_address') is not None

    return tool_uses

# Test your implementation:
test_reg = ToolRegistry()
print('=== Smart Tool Selector Test ===')
expected = {'FC-2024-001': 4, 'FC-2024-002': 3, 'FC-2024-004': 4}

for cid in ['FC-2024-001', 'FC-2024-002', 'FC-2024-004']:
    cust = dict(CUSTOMERS[cid])
    cust['customer_id'] = cid
    selected = smart_tool_selector(cid, cust, test_reg)
    exp = expected.get(cid, '?')
    check = '✅' if len(selected) == exp else '❌'
    print(f'  {check} {cid}: {len(selected)} tools selected (expected {exp})')
    for tu in selected:
        print(f'       → {tu.name}')

In [ ]:
# ─── THAM KHẢO (uncomment để xem solution) ────────────────────────────────────

# def smart_tool_selector_solution(customer_id: str, customer_data: dict, reg: ToolRegistry):
#     tool_uses = []
#     id_number = customer_data['id_number']
#
#     # Rule 4: check compliance cache first
#     comp_cache_key = f"check_compliance:(('customer_id', '{customer_id}'), ('id_number', '{id_number}'))"
#     cached_comp = reg._get_cache(comp_cache_key)
#     if cached_comp and cached_comp.get('blacklisted'):
#         # Blacklisted — return only compliance with is_error simulation
#         return [ToolUseBlock('tu_compliance_cached', 'check_compliance',
#                              {'customer_id': customer_id, 'id_number': id_number})]
#
#     # Rule 1: always call critical tools
#     tool_uses.append(ToolUseBlock('tu_credit', 'get_credit_score',
#                                   {'customer_id': customer_id, 'id_number': id_number}))
#     tool_uses.append(ToolUseBlock('tu_comp', 'check_compliance',
#                                   {'customer_id': customer_id, 'id_number': id_number}))
#
#     # Rule 2: income if declared > 0
#     if customer_data.get('monthly_income', 0) > 0:
#         tool_uses.append(ToolUseBlock('tu_income', 'verify_income',
#                                       {'customer_id': customer_id,
#                                        'declared_income': customer_data['monthly_income']}))
#
#     # Rule 3: property if address provided
#     if customer_data.get('property_address'):
#         tool_uses.append(ToolUseBlock('tu_prop', 'get_property_valuation',
#                                       {'address': customer_data['property_address']}))
#     return tool_uses

print('Uncomment solution block above để xem đáp án.')